# Weather Hazard Category · Multinomial Logistic Regression

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts WHC exactly +7 days ahead*

This notebook trains and evaluates the **Multinomial Logistic Regression** model for the **Weather Hazard Category (WHC)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

A regularised multinomial logistic regression will be **competitive with the tree ensembles** on
macro F1. WHI is a *linear* weighted sum of five hazard scores and the classes are thresholds on
that sum, so a linear decision boundary is close to the correct form, a specific reason to expect
a linear model to do well.

The competing possibility is that the relationship between today's features and the class
*seven days ahead* is not linear even though the same-day one is, in which case the trees win.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `StandardScaler` → `LogisticRegression` (multinomial) | Interpretable coefficients and calibrated probabilities |
| Imbalance | `class_weight` tuned over none and balanced | The linear equivalent of the ensembles' class weighting |
| Strategy | One classifier for the single +7 day horizon | Matches the deployed artefact layout |
| Selection | Optuna TPE sampler, 20 trials per target | See section 6 |
| Objective | validation **macro F1** (higher is better) | Used for every comparison |

### Workflow

All ten experiment notebooks run the same code from `nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model specification in section 1 differs between notebooks.**

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Default configuration | An untuned reference point |
| 5 | Feature-family ablation | What each block of engineered features contributes |
| 6 | Hyperparameter selection | Choose settings on the validation split |
| 7 | Seed stability | Is the tuning gain real or noise? |
| 8 | Final model | Final model against the baselines |
| 9 | Feature importance | Which features the model relies on |
| 10 | Generalisation check | Training vs validation performance |
| 11 | Save artefacts | Model bundle + `/model-metadata` document |
| 12 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression

from nucleus_ml import CategoricalParameter, FloatParameter
from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "weather_hazard"
EXPERIMENT = "whc_logistic_regression"
ALGORITHM = "Multinomial Logistic Regression"
N_TRIALS = 20  # Optuna trials per target

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It creates the experiment from the estimator class, which the package builds (seeding it and, where requested, scaling the features), so no helper function is needed. It also sets the settings that are never tuned (`fixed`), a sensible untuned configuration (`default_params`), a cheaper configuration for the ablation (`ablation_params`) and the search space for section 6, written with the `nucleus_ml` parameter classes.

`scale=True` standardises the features inside the same pipeline, so the scaler is refitted on
training data only. `C` is the inverse regularisation strength, the only capacity control a
linear model has. It is searched across six orders of magnitude because the feature matrix is
wide and highly collinear, and strong shrinkage is the standard remedy. `max_iter` is generous so
no fit stops early.

In [ ]:
exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    estimator=LogisticRegression,
    fixed={"max_iter": 4000},
    scale=True,
    default_params={"C": 1.0, "class_weight": "balanced"},
    search_space={
        "C": FloatParameter(1e-4, 1e2, log=True),
        "class_weight": CategoricalParameter([None, "balanced"]),
    },
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is
reserved for a single evaluation in the comparison notebook, once one model has been
selected, so nothing in this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up
> filter and an embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Three naive forecasts, all fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Majority class** | Always predict the most common class | Nothing but the class balance |
| **Persistence** | The class in 7 days equals today's class (from today's WHI) | The current weather |
| **Seasonal mode** | The most common class for that month | The season |

The majority-class rule scores a high *accuracy* but a low *macro F1*, which is exactly why
accuracy is not used to select models for this imbalanced target.

In [ ]:
exp.run_baselines()

---
## 4. Default configuration

The untuned default is scored before any search. It is the reference for judging
what tuning bought in section 8; a search that fails to beat a
sensible default is a result worth reporting. `best_baseline` is repeated for
comparison.

In [ ]:
exp.run_defaults()

---
## 5. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column), and the model is refitted at each step. This measures what each block of engineering contributes instead of assuming it. `ablation_params` are used throughout so the differences come from the features, not from tuning. Scores are on validation (macro F1).

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 6. Hyperparameter selection

The objective is validation **macro F1** (higher is better).

**Why macro F1.** It averages F1 across classes without weighting by frequency, so a model that
ignores the rare hazard classes cannot hide behind the common Low Risk days. Accuracy would reward
exactly that behaviour. The average is taken over the classes present in training, because a
class the model never saw cannot be predicted.

**Why validation, not cross-validation.** The split is chronological. Shuffled k-fold would put future days into training and inflate every score, because consecutive days are strongly correlated. The fixed 2024 validation year matches how the model will be used: trained on the past, applied forward.

> **Stated plainly:** selecting on validation makes the validation score slightly optimistic. The honest estimate comes from the test split, which only the comparison notebook reads, once.

In [ ]:
exp.tune(n_trials=N_TRIALS)

In [ ]:
exp.plot_search()
plt.show()

---
## 7. Seed stability

The selected configuration is refitted with three different random seeds. If the
score moves by as much as the spread between the top search candidates
(`search_beats_noise = False`), the search was ranking noise and the conclusion
should say so. Deterministic models show a spread of zero.

In [ ]:
exp.seed_stability()

---
## 8. Final model

The selected configuration is refitted and compared with the untuned `default` configuration and every baseline (it appears as `tuned`). `gain_of_final_model` is positive when the final model is better than that row.

Read the **confusion matrix** as well as the headline score: a model that almost never predicts
the rare classes has not learned to flag hazardous days, whatever its macro F1. The printed
boundary summary shows how many errors sit within a few WHI points of a class threshold; those are
threshold effects rather than genuine misses.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 9. Feature importance

This model has no built-in importance, so **permutation importance** is used: each feature is shuffled on the validation split and the drop in score is recorded. Features with negative importance are candidates for pruning.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature that looks unimportant on its own may belong to an essential block. The ablation in section 5 is the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 10. Generalisation check

Training and validation scores side by side. A small gap means the model
generalises; a large gap means it has memorised patterns in the training years that
do not carry over to 2024.

In [ ]:
exp.generalisation()

---
## 11. Save artefacts

The fitted model(s) are written to
`models/weather_hazard/experiments/whc_logistic_regression/` with:

| File | Contents |
|:--|:--|
| `whc_logistic_regression_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array
> at prediction time and would silently give wrong answers if the columns were
> permuted.

In [ ]:
exp.export()

---
## 12. Conclusions

- **Against the ensembles.** If the linear model is close, recommend it: it is cheaper to serve,
  its coefficients are explainable to stakeholders, and its probabilities are calibrated.
- **The selected `C`.** A very small value means heavy shrinkage, a quantitative statement about
  how much of the 105-column matrix is usable signal at a 7-day lead.
- **Probabilities are the real product.** Returning class probabilities alongside the label would
  let operators set their own alarm threshold — a concrete, defensible API improvement.

> The test split has still not been used. It is scored exactly once, in the `whc_comparison` notebook, after one model has been selected on validation performance across all five WHC experiments.